# Notebook 4: Forecasting Experiments

Covers **Section 4** of the assignment:
- Train LSTM, TCN, Transformer with iterative hyperparameter tuning
- Evaluate on the week December 16–22 for three geographic areas
- Produce all required plots (9 total) and tables (3 total)
- Document all timing statistics

In [6]:
import sys, os, json
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import torch

from data_loader import load_processed
from features import prepare_area_data, make_dataloaders
from models import LSTMForecaster, TCNForecaster, TransformerForecaster
from train import run_experiment, set_seed, get_device
from evaluate import (
    run_full_evaluation, build_results_table,
    plot_predictions, plot_failure_case, compute_all_metrics
)
from tuning import ExperimentLogger

PROCESSED_DIR   = '../data/processed/'
FIGURES_DIR     = '../report/figures/'
EXPERIMENTS_DIR = '../experiments/'
os.makedirs(FIGURES_DIR,     exist_ok=True)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

set_seed(42)
device = get_device()

Using device: cpu


In [7]:
# Load processed matrix and top-3 areas identified in Notebook 02
matrix = load_processed(os.path.join(PROCESSED_DIR, 'traffic_matrix.parquet'))

with open(os.path.join(PROCESSED_DIR, 'top3_areas.json')) as f:
    top3 = json.load(f)['top3']

TARGET_AREAS = top3
PRIMARY_AREA = top3[0]
print('Target areas:', TARGET_AREAS)
print('Primary tuning area:', PRIMARY_AREA)

Loaded processed matrix: (8928, 10000)  from ../data/processed/traffic_matrix.parquet
Target areas: [5161, 5059, 5259]
Primary tuning area: 5161


## 4.1 LSTM — Iterative Hyperparameter Tuning

We tune on the highest-traffic area only, then apply the best config to all three.
Each experiment is logged with its rationale and the reasoning for what to try next.

In [8]:
logger = ExperimentLogger(os.path.join(EXPERIMENTS_DIR, 'experiment_log.jsonl'))

# ============================================================
# LSTM Experiment 1 — baseline
# Rationale: 1-day lookback is the standard starting point in
# network traffic forecasting literature (Huang et al. 2017).
# Hidden=64 and 2 layers match the most common cited config.
# ============================================================
data_lstm1 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
lstm_exp1  = run_experiment(
    model_class  = LSTMForecaster,
    model_kwargs = {'input_size': 1, 'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2},
    data         = data_lstm1,
    n_epochs=100, lr=1e-3, batch_size=64, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='LSTM_exp01',
)

m1 = compute_all_metrics(np.array(lstm_exp1['targets_raw']), np.array(lstm_exp1['preds_raw']))
logger.log(
    experiment_id = 'LSTM_exp01',
    model_name    = 'LSTM',
    config        = {'hidden_size':64,'num_layers':2,'dropout':0.2,'lookback':144,'lr':1e-3,'bs':64},
    metrics       = m1,
    rationale     = 'Baseline from literature. 1-day lookback to capture daily cycle.',
    next_steps    = 'Fill after seeing results: if val loss plateaus early, increase hidden_size or lookback.',
)

Using device: cpu

Model : LSTM_exp01  |  Params: 52,545
  Epoch 001/100  train_loss=0.016815  val_loss=0.000440  lr=1.00e-03  time=18.22s
  Epoch 010/100  train_loss=0.000956  val_loss=0.000256  lr=1.00e-03  time=17.53s
  Epoch 020/100  train_loss=0.000780  val_loss=0.000413  lr=1.00e-03  time=16.99s
  Epoch 030/100  train_loss=0.000732  val_loss=0.000214  lr=5.00e-04  time=16.05s
  Epoch 040/100  train_loss=0.000715  val_loss=0.000280  lr=2.50e-04  time=14.28s
  Early stopping at epoch 40 (best epoch=30)

Training complete.
  Best val loss : 0.000214 (epoch 30)
  Total train   : 757.53s
  Avg per epoch : 18.94s
Result saved → ../experiments/LSTM_exp01_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] LSTM_exp01  (LSTM)
  Config    : {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2, 'lookback': 144, 'lr': 0.001, 'bs': 64}
  Metrics   : MAE=83.5641  RMSE=124.4575  MAPE=8.9%
  Rationale : Baseline from literature. 1-day lookback to capture daily cyc

In [9]:
# ============================================================
# LSTM Experiment 2
# Rationale: ACF from EDA showed significant correlation at
# 2-day lag (288 steps). Expanding lookback + hidden_size to
# give the model more capacity to exploit this structure.
# ============================================================
data_lstm2 = prepare_area_data(matrix, PRIMARY_AREA, lookback=288)
lstm_exp2  = run_experiment(
    model_class  = LSTMForecaster,
    model_kwargs = {'input_size': 1, 'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2},
    data         = data_lstm2,
    n_epochs=100, lr=5e-4, batch_size=64, patience=12,
    output_dir=EXPERIMENTS_DIR, model_name='LSTM_exp02',
)

m2 = compute_all_metrics(np.array(lstm_exp2['targets_raw']), np.array(lstm_exp2['preds_raw']))
logger.log(
    experiment_id = 'LSTM_exp02',
    model_name    = 'LSTM',
    config        = {'hidden_size':128,'num_layers':2,'dropout':0.2,'lookback':288,'lr':5e-4,'bs':64},
    metrics       = m2,
    rationale     = 'ACF showed strong autocorrelation at 2-day lag. Expanding lookback from 144→288.',
    next_steps    = 'Fill after seeing results.',
)

Using device: cpu

Model : LSTM_exp02  |  Params: 207,489
  Epoch 001/100  train_loss=0.014772  val_loss=0.000870  lr=5.00e-04  time=581.66s
  Epoch 010/100  train_loss=0.000918  val_loss=0.000376  lr=5.00e-04  time=58.39s
  Epoch 020/100  train_loss=0.000755  val_loss=0.000271  lr=5.00e-04  time=57.69s
  Epoch 030/100  train_loss=0.000705  val_loss=0.000254  lr=5.00e-04  time=90.17s
  Early stopping at epoch 38 (best epoch=26)

Training complete.
  Best val loss : 0.000227 (epoch 26)
  Total train   : 24685.28s
  Avg per epoch : 649.61s
Result saved → ../experiments/LSTM_exp02_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] LSTM_exp02  (LSTM)
  Config    : {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2, 'lookback': 288, 'lr': 0.0005, 'bs': 64}
  Metrics   : MAE=84.7886  RMSE=128.5023  MAPE=8.6%
  Rationale : ACF showed strong autocorrelation at 2-day lag. Expanding lookback from 144→288.
  Next      : Fill after seeing results.
──────────────

In [10]:
# ============================================================
# LSTM Experiment 3 — refine based on best of exp1/2
# Update this cell's config after seeing exp1+exp2 results.
# ============================================================
data_lstm3 = prepare_area_data(matrix, PRIMARY_AREA, lookback=288)
lstm_exp3  = run_experiment(
    model_class  = LSTMForecaster,
    model_kwargs = {'input_size': 1, 'hidden_size': 128, 'num_layers': 2, 'dropout': 0.3},
    data         = data_lstm3,
    n_epochs=150, lr=5e-4, batch_size=64, patience=15,
    output_dir=EXPERIMENTS_DIR, model_name='LSTM_exp03',
)

m3 = compute_all_metrics(np.array(lstm_exp3['targets_raw']), np.array(lstm_exp3['preds_raw']))
logger.log(
    experiment_id = 'LSTM_exp03',
    model_name    = 'LSTM',
    config        = {'hidden_size':128,'num_layers':2,'dropout':0.3,'lookback':288,'lr':5e-4,'bs':64},
    metrics       = m3,
    rationale     = 'Increasing dropout to 0.3 to reduce overfitting seen in exp02 val curve.',
    next_steps    = 'This is the final LSTM config if val loss improves over exp02.',
)

Using device: cpu

Model : LSTM_exp03  |  Params: 207,489
  Epoch 001/150  train_loss=0.014766  val_loss=0.000875  lr=5.00e-04  time=682.24s
  Epoch 010/150  train_loss=0.000963  val_loss=0.000384  lr=5.00e-04  time=54.38s
  Epoch 020/150  train_loss=0.000758  val_loss=0.000278  lr=5.00e-04  time=56.77s
  Epoch 030/150  train_loss=0.000686  val_loss=0.000249  lr=2.50e-04  time=53.48s
  Epoch 040/150  train_loss=0.000685  val_loss=0.000245  lr=2.50e-04  time=72.57s
  Epoch 050/150  train_loss=0.000662  val_loss=0.000247  lr=2.50e-04  time=70.09s
  Early stopping at epoch 52 (best epoch=37)

Training complete.
  Best val loss : 0.000233 (epoch 37)
  Total train   : 4853.54s
  Avg per epoch : 93.34s
Result saved → ../experiments/LSTM_exp03_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] LSTM_exp03  (LSTM)
  Config    : {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.3, 'lookback': 288, 'lr': 0.0005, 'bs': 64}
  Metrics   : MAE=93.8081  RMSE=133.4560

## 4.2 TCN — Iterative Tuning

In [11]:
# ============================================================
# TCN Experiment 1 — Bai et al. (2018) baseline config
# Receptive field = 1 + (3-1)*2*(1+2+4+8) = 61 steps (~10h)
# ============================================================
data_tcn1 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tcn_exp1  = run_experiment(
    model_class  = TCNForecaster,
    model_kwargs = {'input_size':1,'num_channels':[32,32,64,64],'kernel_size':3,'dropout':0.2},
    data         = data_tcn1,
    n_epochs=100, lr=1e-3, batch_size=64, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='TCN_exp01',
)

mt1 = compute_all_metrics(np.array(tcn_exp1['targets_raw']), np.array(tcn_exp1['preds_raw']))
logger.log(
    experiment_id = 'TCN_exp01',
    model_name    = 'TCN',
    config        = {'channels':[32,32,64,64],'kernel':3,'dropout':0.2,'lookback':144,'lr':1e-3},
    metrics       = mt1,
    rationale     = 'Bai et al. baseline. Receptive field ~61 steps (10h) — shorter than daily cycle.',
    next_steps    = 'Expand receptive field via larger kernel or more blocks if performance is limited.',
)

Using device: cpu

Model : TCN_exp01  |  Params: 55,329
  Epoch 001/100  train_loss=0.005965  val_loss=0.000389  lr=1.00e-03  time=21.65s
  Epoch 010/100  train_loss=0.000801  val_loss=0.000191  lr=1.00e-03  time=21.16s
  Epoch 020/100  train_loss=0.000789  val_loss=0.000184  lr=1.00e-03  time=19.26s
  Epoch 030/100  train_loss=0.000729  val_loss=0.000183  lr=5.00e-04  time=22.45s
  Early stopping at epoch 38 (best epoch=28)

Training complete.
  Best val loss : 0.000177 (epoch 28)
  Total train   : 882.74s
  Avg per epoch : 23.23s
Result saved → ../experiments/TCN_exp01_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] TCN_exp01  (TCN)
  Config    : {'channels': [32, 32, 64, 64], 'kernel': 3, 'dropout': 0.2, 'lookback': 144, 'lr': 0.001}
  Metrics   : MAE=80.9493  RMSE=119.0283  MAPE=8.3%
  Rationale : Bai et al. baseline. Receptive field ~61 steps (10h) — shorter than daily cycle.
  Next      : Expand receptive field via larger kernel or more blocks

In [12]:
# ============================================================
# TCN Experiment 2 — wider kernel, more channels
# RF = 1 + (5-1)*2*(1+2+4+8) = 121 steps (~20h)
# ============================================================
data_tcn2 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tcn_exp2  = run_experiment(
    model_class  = TCNForecaster,
    model_kwargs = {'input_size':1,'num_channels':[64,64,64,64],'kernel_size':5,'dropout':0.2},
    data         = data_tcn2,
    n_epochs=100, lr=1e-3, batch_size=64, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='TCN_exp02',
)

mt2 = compute_all_metrics(np.array(tcn_exp2['targets_raw']), np.array(tcn_exp2['preds_raw']))
logger.log(
    experiment_id = 'TCN_exp02',
    model_name    = 'TCN',
    config        = {'channels':[64,64,64,64],'kernel':5,'dropout':0.2,'lookback':144,'lr':1e-3},
    metrics       = mt2,
    rationale     = 'Wider kernel (3→5) and uniform 64 channels doubles RF to ~20h.',
    next_steps    = 'Fill after results.',
)

Using device: cpu

Model : TCN_exp02  |  Params: 144,897
  Epoch 001/100  train_loss=0.008419  val_loss=0.000363  lr=1.00e-03  time=35.66s
  Epoch 010/100  train_loss=0.000809  val_loss=0.000175  lr=1.00e-03  time=34.36s
  Epoch 020/100  train_loss=0.000786  val_loss=0.000168  lr=1.00e-03  time=35.60s
  Epoch 030/100  train_loss=0.000735  val_loss=0.000259  lr=5.00e-04  time=32.24s
  Early stopping at epoch 31 (best epoch=21)

Training complete.
  Best val loss : 0.000165 (epoch 21)
  Total train   : 1089.70s
  Avg per epoch : 35.15s
Result saved → ../experiments/TCN_exp02_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] TCN_exp02  (TCN)
  Config    : {'channels': [64, 64, 64, 64], 'kernel': 5, 'dropout': 0.2, 'lookback': 144, 'lr': 0.001}
  Metrics   : MAE=80.8713  RMSE=118.2303  MAPE=8.9%
  Rationale : Wider kernel (3→5) and uniform 64 channels doubles RF to ~20h.
  Next      : Fill after results.
───────────────────────────────────────────────────

In [13]:
# TCN Experiment 3 — final refinement
data_tcn3 = prepare_area_data(matrix, PRIMARY_AREA, lookback=288)
tcn_exp3  = run_experiment(
    model_class  = TCNForecaster,
    model_kwargs = {'input_size':1,'num_channels':[64,64,64,64],'kernel_size':5,'dropout':0.3},
    data         = data_tcn3,
    n_epochs=150, lr=5e-4, batch_size=64, patience=15,
    output_dir=EXPERIMENTS_DIR, model_name='TCN_exp03',
)

mt3 = compute_all_metrics(np.array(tcn_exp3['targets_raw']), np.array(tcn_exp3['preds_raw']))
logger.log(
    experiment_id = 'TCN_exp03',
    model_name    = 'TCN',
    config        = {'channels':[64,64,64,64],'kernel':5,'dropout':0.3,'lookback':288,'lr':5e-4},
    metrics       = mt3,
    rationale     = 'Extended lookback to 2 days; reducing LR for finer convergence.',
    next_steps    = 'Final TCN config.',
)

Using device: cpu

Model : TCN_exp03  |  Params: 144,897
  Epoch 001/150  train_loss=0.009480  val_loss=0.000547  lr=5.00e-04  time=64.85s
  Epoch 010/150  train_loss=0.000795  val_loss=0.000250  lr=5.00e-04  time=72.99s
  Epoch 020/150  train_loss=0.000731  val_loss=0.000256  lr=5.00e-04  time=70.54s
  Epoch 030/150  train_loss=0.000706  val_loss=0.000198  lr=5.00e-04  time=61.50s
  Epoch 040/150  train_loss=0.000692  val_loss=0.000205  lr=2.50e-04  time=80.20s
  Epoch 050/150  train_loss=0.000671  val_loss=0.000219  lr=1.25e-04  time=100.59s
  Early stopping at epoch 53 (best epoch=38)

Training complete.
  Best val loss : 0.000176 (epoch 38)
  Total train   : 3870.48s
  Avg per epoch : 73.02s
Result saved → ../experiments/TCN_exp03_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] TCN_exp03  (TCN)
  Config    : {'channels': [64, 64, 64, 64], 'kernel': 5, 'dropout': 0.3, 'lookback': 288, 'lr': 0.0005}
  Metrics   : MAE=82.1206  RMSE=120.4585  MAPE=8

## 4.3 Transformer — Iterative Tuning

In [14]:
# ============================================================
# Transformer Exp 1 — conservative start
# Pre-LN + low LR to prevent divergence (known instability
# of Transformers with high learning rates on small datasets).
# ============================================================
data_tf1 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tf_exp1  = run_experiment(
    model_class  = TransformerForecaster,
    model_kwargs = {'input_size':1,'d_model':64,'nhead':4,'num_layers':2,'dim_feedfwd':128,'dropout':0.1},
    data         = data_tf1,
    n_epochs=100, lr=1e-4, batch_size=32, patience=10,
    output_dir=EXPERIMENTS_DIR, model_name='Transformer_exp01',
)

mf1 = compute_all_metrics(np.array(tf_exp1['targets_raw']), np.array(tf_exp1['preds_raw']))
logger.log(
    experiment_id = 'TF_exp01',
    model_name    = 'Transformer',
    config        = {'d_model':64,'nhead':4,'layers':2,'ffn':128,'dropout':0.1,'lookback':144,'lr':1e-4},
    metrics       = mf1,
    rationale     = 'Small conservative config with pre-LN. Low LR to avoid early divergence.',
    next_steps    = 'If stable: scale d_model or add layers. If slow to converge: try 5e-4.',
)

Using device: cpu

Model : Transformer_exp01  |  Params: 69,185
  Epoch 001/100  train_loss=0.018297  val_loss=0.000338  lr=1.00e-04  time=88.01s
  Epoch 010/100  train_loss=0.001449  val_loss=0.000267  lr=1.00e-04  time=86.74s
  Early stopping at epoch 17 (best epoch=7)

Training complete.
  Best val loss : 0.000258 (epoch 7)
  Total train   : 1527.06s
  Avg per epoch : 89.83s
Result saved → ../experiments/Transformer_exp01_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] TF_exp01  (Transformer)
  Config    : {'d_model': 64, 'nhead': 4, 'layers': 2, 'ffn': 128, 'dropout': 0.1, 'lookback': 144, 'lr': 0.0001}
  Metrics   : MAE=151.8728  RMSE=191.9296  MAPE=26.8%
  Rationale : Small conservative config with pre-LN. Low LR to avoid early divergence.
  Next      : If stable: scale d_model or add layers. If slow to converge: try 5e-4.
───────────────────────────────────────────────────────


In [15]:
# Transformer Exp 2 — scale up if exp1 was stable
data_tf2 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tf_exp2  = run_experiment(
    model_class  = TransformerForecaster,
    model_kwargs = {'input_size':1,'d_model':128,'nhead':4,'num_layers':3,'dim_feedfwd':256,'dropout':0.1},
    data         = data_tf2,
    n_epochs=30, lr=5e-4, batch_size=32, patience=5,
    output_dir=EXPERIMENTS_DIR, model_name='Transformer_exp02',
)

mf2 = compute_all_metrics(np.array(tf_exp2['targets_raw']), np.array(tf_exp2['preds_raw']))
logger.log(
    experiment_id = 'TF_exp02',
    model_name    = 'Transformer',
    config        = {'d_model':128,'nhead':4,'layers':3,'ffn':256,'dropout':0.1,'lookback':144,'lr':5e-4},
    metrics       = mf2,
    rationale     = 'Scaling d_model 64→128, 2→3 layers. Slightly higher LR given stable exp01.',
    next_steps    = 'Fill after results.',
)

Using device: cpu

Model : Transformer_exp02  |  Params: 406,017
  Epoch 001/30  train_loss=0.008728  val_loss=0.002948  lr=5.00e-04  time=187.98s
  Epoch 010/30  train_loss=0.000872  val_loss=0.000332  lr=2.50e-04  time=199.16s
  Early stopping at epoch 18 (best epoch=13)

Training complete.
  Best val loss : 0.000293 (epoch 13)
  Total train   : 3587.07s
  Avg per epoch : 199.27s
Result saved → ../experiments/Transformer_exp02_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] TF_exp02  (Transformer)
  Config    : {'d_model': 128, 'nhead': 4, 'layers': 3, 'ffn': 256, 'dropout': 0.1, 'lookback': 144, 'lr': 0.0005}
  Metrics   : MAE=109.9947  RMSE=154.7993  MAPE=13.5%
  Rationale : Scaling d_model 64→128, 2→3 layers. Slightly higher LR given stable exp01.
  Next      : Fill after results.
───────────────────────────────────────────────────────


In [17]:
# Transformer Exp 3 — final
# Note: lookback reduced 288→144 to manage O(L²) attention cost on CPU.
# d_model and layers kept the same to test the larger architecture.
data_tf3 = prepare_area_data(matrix, PRIMARY_AREA, lookback=144)
tf_exp3  = run_experiment(
    model_class  = TransformerForecaster,
    model_kwargs = {'input_size':1,'d_model':128,'nhead':4,'num_layers':3,'dim_feedfwd':256,'dropout':0.2},
    data         = data_tf3,
    n_epochs=30, lr=5e-4, batch_size=32, patience=5,
    output_dir=EXPERIMENTS_DIR, model_name='Transformer_exp03',
)

mf3 = compute_all_metrics(np.array(tf_exp3['targets_raw']), np.array(tf_exp3['preds_raw']))
logger.log(
    experiment_id = 'TF_exp03',
    model_name    = 'Transformer',
    config        = {'d_model':128,'nhead':4,'layers':3,'ffn':256,'dropout':0.2,'lookback':144,'lr':5e-4},
    metrics       = mf3,
    rationale     = 'Larger architecture (d_model=128, 3 layers). Lookback kept at 144 to avoid O(L²) attention cost on CPU.',
    next_steps    = 'Final Transformer config.',
)

# Print full tuning summary
logger.print_summary()

Using device: cpu

Model : Transformer_exp03  |  Params: 406,017
  Epoch 001/30  train_loss=0.008746  val_loss=0.002493  lr=5.00e-04  time=209.53s
  Epoch 010/30  train_loss=0.001039  val_loss=0.000905  lr=2.50e-04  time=182.01s
  Early stopping at epoch 14 (best epoch=9)

Training complete.
  Best val loss : 0.000329 (epoch 9)
  Total train   : 2709.20s
  Avg per epoch : 193.50s
Result saved → ../experiments/Transformer_exp03_sq5161_result.json

───────────────────────────────────────────────────────
[Logged] TF_exp03  (Transformer)
  Config    : {'d_model': 128, 'nhead': 4, 'layers': 3, 'ffn': 256, 'dropout': 0.2, 'lookback': 144, 'lr': 0.0005}
  Metrics   : MAE=127.3610  RMSE=178.6444  MAPE=14.6%
  Rationale : Larger architecture (d_model=128, 3 layers). Lookback kept at 144 to avoid O(L²) attention cost on CPU.
  Next      : Final Transformer config.
───────────────────────────────────────────────────────

Experiment Summary
experiment_id  model_name  metrics.MAE  metrics.RMSE  met

## 4.4 Final Models — Evaluate on All Three Areas

**Before running this cell**: update the `BEST_*` dicts below with the config
that achieved the lowest validation MAE in sections 4.1–4.3 above.

In [ ]:
# ── UPDATE THESE after reviewing tuning results ──────────────────────────────
BEST_LSTM = {
    'model_kwargs': {'input_size':1,'hidden_size':128,'num_layers':2,'dropout':0.3},
    'lr': 5e-4, 'batch_size': 64, 'lookback': 288,
}
BEST_TCN = {
    'model_kwargs': {'input_size':1,'num_channels':[64,64,64,64],'kernel_size':5,'dropout':0.3},
    'lr': 5e-4, 'batch_size': 64, 'lookback': 288,
}
BEST_TF = {
    'model_kwargs': {'input_size':1,'d_model':128,'nhead':4,'num_layers':3,'dim_feedfwd':256,'dropout':0.2},
    'lr': 5e-4, 'batch_size': 32, 'lookback': 288,
}
# ─────────────────────────────────────────────────────────────────────────────

MODEL_CONFIGS = [
    ('LSTM',        LSTMForecaster,        BEST_LSTM),
    ('TCN',         TCNForecaster,         BEST_TCN),
    ('Transformer', TransformerForecaster, BEST_TF),
]

all_results = {}  # {square_id: {model_name: result}}

for sq_id in TARGET_AREAS:
    print(f'\n{"="*60}')
    print(f'Final evaluation  —  Square {sq_id}')
    print(f'{"="*60}')
    area_results = {}

    for model_name, model_class, cfg in MODEL_CONFIGS:
        data = prepare_area_data(matrix, sq_id, lookback=cfg['lookback'])
        result = run_experiment(
            model_class  = model_class,
            model_kwargs = cfg['model_kwargs'],
            data         = data,
            n_epochs     = 150,
            lr           = cfg['lr'],
            batch_size   = cfg['batch_size'],
            patience     = 15,
            output_dir   = EXPERIMENTS_DIR,
            model_name   = f'{model_name}_final_sq{sq_id}',
        )
        area_results[model_name] = result

    all_results[sq_id] = area_results

print('\nAll final experiments complete.')

## 4.5 Results Tables and Plots

Generates:
- **3 results tables** (one per area): MAE, MAPE, RMSE, train time, inference time
- **9 prediction plots** (one per model per area): actual vs predicted Dec 16–22
- **1 failure case plot** per area
- **Training curve plots**

In [ ]:
combined_metrics = run_full_evaluation(all_results, save_dir=FIGURES_DIR)
print('\nCombined Results Table:')
combined_metrics

In [ ]:
# Final summary of timing statistics (Section 4 item IV)
print('Timing Summary')
print('=' * 60)
for sq_id, area_res in all_results.items():
    print(f'\nSquare {sq_id}:')
    for mname, res in area_res.items():
        h = res['history']
        print(f'  {mname:<15} '
              f"train={h['total_train_time_s']:.1f}s  "
              f"({h['epochs_run']} epochs, best={h['best_epoch']})  "
              f"inf={res['inference_time_per_sample_ms']:.3f}ms/sample")